# Qwen2.5-7B CodeContest Eval on TPU

Loads Qwen2.5-7B-Instruct via the tunix JAX stack, generates Python solutions
for CodeContest problems, and evaluates them by running the code against the
provided test cases.

Code execution uses threads (no `fork`, no `subprocess`) so it is safe to run
after JAX/TPU initialisation.

In [ ]:
# ── paths ──────────────────────────────────────────────────────────────────
MODEL_PATH = "/path/to/qwen2.5-7b-instruct"  # local safetensors directory
DATA_PATH  = "/path/to/codecontest.json"      # JSON list of problem dicts

# ── TPU topology ───────────────────────────────────────────────────────────
TP_SIZE = 8   # tensor-parallel chips; use 1 for a single chip

# ── generation ─────────────────────────────────────────────────────────────
K_CODE         = 4      # code samples per problem (pass@K)
MAX_NEW_TOKENS = 1024
MAX_CACHE_LEN  = 8192   # KV-cache slots; must be >= max_prompt_len + MAX_NEW_TOKENS
MAX_PROMPT_LEN = MAX_CACHE_LEN - MAX_NEW_TOKENS  # derived: max tokens a prompt may use
TEMPERATURE    = 0.8
TOP_P          = 0.95

# ── evaluation ─────────────────────────────────────────────────────────────
MAX_TEST     = 5    # max test cases to check per problem
EXEC_TIMEOUT = 5.0  # seconds per test case

In [ ]:
import io
import json
import re
import sys
import threading
import typing

import jax
import jax.numpy as jnp
import numpy as np
from flax import nnx
from jax.sharding import Mesh
from transformers import AutoTokenizer

from tunix.generate import sampler as sampler_lib
from tunix.models.qwen2 import model as model_lib
from tunix.models.qwen2 import params as params_lib

print(f"JAX backend : {jax.default_backend()}")
print(f"JAX devices : {jax.devices()}")

## Load model

In [ ]:
devices      = jax.devices()
mesh_devices = np.array(devices[:TP_SIZE]).reshape(1, TP_SIZE)
mesh         = Mesh(mesh_devices, axis_names=("fsdp", "tp"))

tokenizer    = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

model_config = model_lib.ModelConfig.qwen2p5_7b_instruct()
model        = params_lib.create_model_from_safe_tensors(
    file_dir=MODEL_PATH,
    config=model_config,
    mesh=mesh,
    dtype=jnp.bfloat16,
)

sampler = sampler_lib.Sampler(
    model,
    tokenizer,
    sampler_lib.CacheConfig(
        cache_size=MAX_CACHE_LEN,
        num_layers=model_config.num_layers,
        num_kv_heads=model_config.num_kv_heads,
        head_dim=model_config.head_dim,
    ),
)
print("Model and sampler ready.")

## Load data

In [ ]:
# Expected fields per problem dict:
#   question        : str
#   test_input      : list[str]
#   test_output     : list[str]
#   test_time_limit : float  (optional, defaults to 2.0)
with open(DATA_PATH) as f:
    data = json.load(f)

print(f"Loaded {len(data)} problems.")

In [ ]:
# ── GT solutions (used to evaluate tester accuracy in both eval sections) ─────
GT_SOLUTIONS_PATH = "/path/to/py3_solutions_test.jsonl"
with open(GT_SOLUTIONS_PATH) as _f:
    _sol_recs = [json.loads(l) for l in _f]
gt_solutions = {
    str(r["task_id"]): r["py3_solution"]
    for r in _sol_recs
    if r["source"] != "failed" and r["py3_solution"] is not None
}
print(f"GT solutions loaded: {len(gt_solutions)}")

## Prompt builder + code extractor

In [ ]:
from iterative_eval import CODE_PROMPT_TEMPLATE, extract_code

CASE_PROMPT_TEMPLATE = (
    "<|im_start|>You are a helpful assistant help user generate test examples for coding tasks. "
    "<|im_end|>\n<|im_start|>User: "
    "Given a coding task, instead of providing the final script, your task is to generate a new test example "
    "(both input, output and explanation).\n"
    "This is the problem:\n{problem}\n\n"
    "{example_intro}\n"
    "You need to provide a new test example. A good test example should be completely accurate and conform to "
    "the problem's format requirements, while also possessing enough discriminative power to distinguish correct "
    "code from incorrect code.\n"
    "Before providing a test example, you must think carefully and reason step by step to derive an input and "
    "output you are very confident are correct. For example, start by designing an input you can reliably handle, "
    "then compute the output step by step. If you're unsure about the output, revise or re-design the input to "
    "ensure accuracy. Directly providing input/output pairs without this process is discouraged, as it often "
    "results in low accuracy.\n"
    "Finally, after completing these previous thinking and derivation steps (you should not write the final test "
    "example unless you have gone through these steps very thoroughly), you MUST put your final test example in "
    "the following format:\n\n"
    "**Test Input:**\n```input here```\n\n"
    "**Test Output:**\n```output here```\n\n"
    "**Explanation:**\n\nexplanation here.\n "
    "<|im_end|>\n<|im_start|>Assistant: "
)


def build_prompt(problem: str) -> str:
    return CODE_PROMPT_TEMPLATE.format(problem=problem)

## Code execution (thread-based, TPU-safe)

We run generated code in a daemon thread rather than a subprocess to avoid
forking the process after JAX/TPU initialisation. The thread shares the
interpreter but runs in an isolated `exec` context with its own `input`/stdout.

In [ ]:
import iterative_eval
from iterative_eval import run_code, outputs_match

# If you edit iterative_eval.py, re-run this cell after the reload in the iterative-eval section.


def evaluate_code(code, test_inputs, test_outputs, time_limit):
    n = min(len(test_inputs), MAX_TEST)
    passed = sum(
        outputs_match(run_code(code, inp, timeout=time_limit), exp)
        for inp, exp in zip(test_inputs[:n], test_outputs[:n])
    )
    return {"passed": passed, "total": n}

## Smoke-test: code execution harness

Run this cell to verify `run_code` / `evaluate_code` work correctly **before**
loading the model. Tests four scenarios: correct code, wrong output, runtime
error, and infinite loop (timeout).

In [ ]:
import random

# ── 20 test cases for "read a and b, print a+b" ───────────────────────────
rng = random.Random(42)
_pairs         = [(rng.randint(-1000, 1000), rng.randint(-1000, 1000)) for _ in range(20)]
_smoke_inputs  = [f"{a} {b}" for a, b in _pairs]
_smoke_outputs = [str(a + b)  for a, b in _pairs]

# ── four candidate solutions ───────────────────────────────────────────────
_cases = {
    "correct": ("""\
a, b = map(int, input().split())
print(a + b)
""", _smoke_inputs, _smoke_outputs),
    "wrong output": ("""\
a, b = map(int, input().split())
print(a - b)
""", _smoke_inputs, _smoke_outputs),
    "runtime error": ("""\
a, b = map(int, input().split())
print(1 / 0)
""", _smoke_inputs, _smoke_outputs),
    # Only 1 timeout case — each waits the full timeout, no need to repeat 20x
    "timeout": ("""\
a, b = map(int, input().split())
while True:
    pass
""", _smoke_inputs[:1], _smoke_outputs[:1]),
}

# ── run and report ─────────────────────────────────────────────────────────
print(f"{'case':<16} {'passed':>6}  {'total':>5}  per-test results")
print("-" * 68)

all_ok = True
for label, (code, inputs, outputs) in _cases.items():
    per_test = []
    for inp, exp in zip(inputs, outputs):
        out = run_code(code.strip(), inp, timeout=1.0)
        per_test.append("P" if outputs_match(out, exp) else "F")
    passed = per_test.count("P")
    total  = len(per_test)
    print(f"{label:<16} {passed:>6}  {total:>5}  {''.join(per_test)}")

    if label == "correct"      and passed != total: all_ok = False
    if label == "wrong output" and passed == total: all_ok = False
    if label == "timeout"      and passed == total: all_ok = False

print()
print("Harness OK — safe to proceed." if all_ok else "HARNESS ISSUE — check run_code / outputs_match.")

In [ ]:
# NOTE: subprocess-based run_code is cleaner (no zombie threads, process-level
# isolation for MemoryError/infinite loops) but sys.executable is None in this
# kernel, so subprocess.run fails.  Thread-based run_code (cell 71780324) is
# used for now.  Migrate to subprocess when running outside the notebook.

## Generate and evaluate

In [ ]:
PROBLEMS = data[:10]  # adjust slice as needed

all_results = []
skipped = 0

for prob_idx, prob in enumerate(PROBLEMS):
    question    = prob["question"]
    test_inputs = prob["test_input"]
    test_outputs= prob["test_output"]
    time_limit  = float(prob.get("test_time_limit", 2.0))

    prompt     = build_prompt(question)
    prompt_len = len(tokenizer.encode(prompt))

    if prompt_len > MAX_PROMPT_LEN:
        print(f"[{prob_idx+1:3d}/{len(PROBLEMS)}] SKIP  prompt too long ({prompt_len} tokens > {MAX_PROMPT_LEN})")
        all_results.append({"passed": 0, "total": 0, "code": None, "skipped": True})
        skipped += 1
        continue

    out = sampler(
        input_strings=[prompt] * K_CODE,
        max_generation_steps=MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
        top_p=TOP_P,
    )

    best = {"passed": 0, "total": 0, "code": None, "skipped": False}
    for gen_text in out.text:
        code = extract_code(gen_text)
        if code is None:
            continue
        score = evaluate_code(code, test_inputs, test_outputs, time_limit)
        if score["passed"] > best["passed"]:
            best = {**score, "code": code, "skipped": False}

    all_results.append(best)
    solved = best["total"] > 0 and best["passed"] == best["total"]
    marker = "PASS" if solved else "FAIL"
    print(f"[{prob_idx+1:3d}/{len(PROBLEMS)}] {marker}  {best['passed']}/{best['total']} tests  (prompt {prompt_len} tok)")

## Summary

In [ ]:
attempted = [r for r in all_results if not r.get("skipped")]
solved    = sum(1 for r in attempted if r["total"] > 0 and r["passed"] == r["total"])
print(f"pass@{K_CODE}  {solved}/{len(attempted)}  ({solved/len(attempted):.1%})  ({skipped} skipped — prompt too long)")

## Simple test: Maximum Subarray Sum

A single hand-crafted problem in CodeContest format to verify the full pipeline
(prompt → model → code extraction → execution → pass/fail).

**Problem:** Given a sequence of N integers, find the maximum sum of any contiguous subarray.

*Why this problem?* Not trivial (requires Kadane's algorithm or similar), but well within the reach of a 7B model. It has clear stdin/stdout, deterministic outputs, and edge cases (all-negative array, single element).

In [ ]:
# ── Simple test problem (CodeContest format) ──────────────────────────────
#
# Maximum Subarray Sum
# --------------------
# Given N integers, output the maximum sum of any contiguous subarray.
# Requires Kadane's algorithm; edge-cases: all-negative, single element.

SIMPLE_PROBLEM = {
    "question": """\
Given a sequence of N integers, find the maximum sum of any contiguous subarray.

Input:
  The first line contains a single integer N (1 ≤ N ≤ 100 000).
  The second line contains N space-separated integers a_1, a_2, ..., a_N
  (-10 000 ≤ a_i ≤ 10 000).

Output:
  Print a single integer: the maximum subarray sum.

Examples:
  Input          Output
  5              4
  -2 1 -3 4 -1

  Input          Output
  4              10
  1 2 3 4

  Input          Output
  3              -1
  -1 -2 -3
""",
    "test_input": [
        "1\n5",
        "5\n-2 1 -3 4 -1",
        "4\n1 2 3 4",
        "3\n-1 -2 -3",
        "6\n-2 -3 4 -1 -2 1",
        "8\n2 -1 2 3 4 -5 2 1",
        "5\n-5 -4 -3 -2 -1",
        "6\n1 -1 1 -1 1 1",
        "4\n100 -50 100 -50",
        "7\n-1 5 -2 3 -1 2 -1",
    ],
    "test_output": [
        "5",
        "4",
        "10",
        "-1",
        "4",
        "10",
        "-1",
        "2",
        "150",
        "7",
    ],
    "test_time_limit": 2.0,
}

# ── Generate K_CODE solutions from the model ──────────────────────────────
prompt = build_prompt(SIMPLE_PROBLEM["question"])
out = sampler(
    input_strings=[prompt] * K_CODE,
    max_generation_steps=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
)

# ── Evaluate each sample, keep the best ───────────────────────────────────
test_inputs  = SIMPLE_PROBLEM["test_input"]
test_outputs = SIMPLE_PROBLEM["test_output"]
time_limit   = SIMPLE_PROBLEM["test_time_limit"]
n_tests      = len(test_inputs)

best = {"passed": -1, "total": n_tests, "code": None, "text": None}

print(f"{'sample':>7}  {'passed':>7}  {'results'}")
print("-" * 50)

for i, gen_text in enumerate(out.text):
    code = extract_code(gen_text)
    if code is None:
        print(f"{i:>7}  {'no code':>7}")
        continue
    per_test = []
    for inp, exp in zip(test_inputs, test_outputs):
        actual = run_code(code, inp, timeout=time_limit)
        per_test.append("P" if outputs_match(actual, exp) else "F")
    passed = per_test.count("P")
    print(f"{i:>7}  {passed:>3}/{n_tests:<3}   {''.join(per_test)}")
    if passed > best["passed"]:
        best = {"passed": passed, "total": n_tests, "code": code, "text": gen_text}

# ── Final verdict ─────────────────────────────────────────────────────────
print()
solved = best["passed"] == best["total"] and best["total"] > 0
if solved:
    print(f"TASK SUCCESS  — best sample passed {best['passed']}/{best['total']} tests")
else:
    print(f"TASK FAILED   — best sample passed {best['passed']}/{best['total']} tests")

if best["code"]:
    print("\n── Best solution ──────────────────────────────────────────────────────")
    print(best["code"])

## Unit-test eval

Given a problem statement, can the model generate *valid unit tests* (new input/output pairs)?

Pipeline:
1. For each problem, generate **K_CASE** unit test candidates with the model.
2. Also generate **K_CODE** code solutions (same as the main eval).
3. Build two boolean matrices per problem:
   - `test_bool_table` (K_CODE × n_gt): each code vs each **ground-truth** test case.
   - `case_bool_table` (K_CODE × K_CASE): each code vs each **generated** test case.
4. Metrics:
   - **code_acc** — fraction of problems where ≥1 generated code passes all ground-truth tests.
   - **BoN_acc** — use `case_bool_table` to pick the best code; check it on ground-truth tests. Measures whether the generated unit tests are good enough to identify the best solution.
   - **case_quality** — among correct code solutions, what fraction of generated unit tests do they pass? (High → generated tests are valid; low → tests have wrong answers.)

In [ ]:
from iterative_eval import _normalise

K_CASE     = 16    # unit-test candidates per problem (paper default)
NO_EXAMPLE = True  # match eval_config_sample.py: don't feed public examples to the model


def build_case_prompt(
    problem: str,
    example_inputs:  list | None = None,
    example_outputs: list | None = None,
) -> str:
    """Build the unit-test-generation prompt using the reference template."""
    if NO_EXAMPLE:
        example_intro = " "
    else:
        n = min(len(example_inputs or []), len(example_outputs or []))
        if n == 0:
            example_intro = " "
        elif n == 1:
            example_intro = (
                f"We already have one test sample:\n"
                f" Its input is {example_inputs[0]!r}. Its output is {example_outputs[0]!r}.\n"
            )
        else:
            inp_repr = ", ".join(repr(x) for x in example_inputs[:n])
            out_repr = ", ".join(repr(x) for x in example_outputs[:n])
            example_intro = (
                f"We already have {n} test samples:\n"
                f" The inputs are, respectively, {inp_repr}. "
                f"The corresponding outputs are {out_repr}.\n"
            )
    return CASE_PROMPT_TEMPLATE.format(problem=problem, example_intro=example_intro)


def extract_test_case(text: str) -> tuple:
    """Extract (test_input, test_output) from a model response, or (None, None)."""
    def _find(pattern_backtick, pattern_plain, text):
        m = re.findall(pattern_backtick, text, re.DOTALL)
        if m:
            return _normalise(m[-1].lstrip("\n"))
        m = re.findall(pattern_plain, text, re.DOTALL)
        return _normalise(m[-1].strip()) if m else None

    inp = _find(
        r'\*\*Test Input:\*\*\s*```(.*?)```',
        r'\*\*Test Input:\*\*\s*([\s\S]*?)(?=\*\*Test Output:\*\*)',
        text,
    )
    out = _find(
        r'\*\*Test Output:\*\*\s*```(.*?)```',
        r'\*\*Test Output:\*\*\s*([\s\S]*?)(?=\*\*Explanation:|\*\*Test Input:|$)',
        text,
    )
    return inp, out

In [ ]:
import json
import os

UT_PROBLEMS = data[:10]   # adjust slice as needed

# ── checkpoint ────────────────────────────────────────────────────────────
CHECKPOINT_PATH = "/tmp/ut_checkpoint.json"
# Add problem indices (0-based) here to hard-skip them on next run.
# Use this after a crash caused by a specific problem eating all RAM.
FORCE_SKIP = set()  # e.g. {3, 7}

def _load_checkpoint(path):
    if path and os.path.exists(path):
        with open(path) as f:
            return {int(k): v for k, v in json.load(f).items()}
    return {}

def _save_checkpoint(path, ckpt):
    if path:
        with open(path, "w") as f:
            json.dump(ckpt, f)

checkpoint = _load_checkpoint(CHECKPOINT_PATH)
print(f"Checkpoint loaded: {len(checkpoint)} problem(s) already done.")

ut_results = [None] * len(UT_PROBLEMS)
ut_skipped = 0

for prob_idx, prob in enumerate(UT_PROBLEMS):
    # ── force-skip (e.g. known OOM problem) ──────────────────────────
    if prob_idx in FORCE_SKIP and prob_idx not in checkpoint:
        print(f"[{prob_idx+1:3d}/{len(UT_PROBLEMS)}] FORCE-SKIP")
        ut_skipped += 1
        checkpoint[prob_idx] = None
        _save_checkpoint(CHECKPOINT_PATH, checkpoint)
        continue

    # ── resume ────────────────────────────────────────────────────────
    if prob_idx in checkpoint:
        entry = checkpoint[prob_idx]
        if entry is None:
            ut_skipped += 1
        else:
            ut_results[prob_idx] = {
                "codes":           entry["codes"],
                "gen_cases":       [tuple(x) for x in entry["gen_cases"]],
                "test_bool_table": entry["test_bool_table"],
                "case_bool_table": entry["case_bool_table"],
            }
        print(f"[{prob_idx+1:3d}/{len(UT_PROBLEMS)}] RESUMED")
        continue

    question         = prob["question"]
    test_inputs      = prob["test_input"]
    test_outputs     = prob["test_output"]
    time_limit       = float(prob.get("test_time_limit", 2.0))
    example_inputs   = prob.get("example_input",  [])
    example_outputs  = prob.get("example_output", [])

    # ── guard: prompt length ──────────────────────────────────────────
    code_prompt     = build_prompt(question)
    code_prompt_len = len(tokenizer.encode(code_prompt))
    case_prompt     = build_case_prompt(question, example_inputs, example_outputs)
    case_prompt_len = len(tokenizer.encode(case_prompt))

    if code_prompt_len > MAX_PROMPT_LEN or case_prompt_len > MAX_PROMPT_LEN:
        which  = "code" if code_prompt_len > MAX_PROMPT_LEN else "case"
        length = code_prompt_len if code_prompt_len > MAX_PROMPT_LEN else case_prompt_len
        print(f"[{prob_idx+1:3d}/{len(UT_PROBLEMS)}] SKIP  {which} prompt too long ({length} tok)")
        ut_skipped += 1
        checkpoint[prob_idx] = None
        _save_checkpoint(CHECKPOINT_PATH, checkpoint)
        continue

    # ── generate K_CODE code solutions ───────────────────────────────
    code_out = sampler(
        input_strings=[code_prompt] * K_CODE,
        max_generation_steps=MAX_NEW_TOKENS,
        max_prompt_length=MAX_PROMPT_LEN,
        temperature=TEMPERATURE,
        top_p=TOP_P,
    )
    codes = [extract_code(t) for t in code_out.text]

    # ── generate K_CASE unit tests ────────────────────────────────────
    case_out = sampler(
        input_strings=[case_prompt] * K_CASE,
        max_generation_steps=MAX_NEW_TOKENS,
        max_prompt_length=MAX_PROMPT_LEN,
        temperature=TEMPERATURE,
        top_p=TOP_P,
    )
    gen_cases = [extract_test_case(t) for t in case_out.text]
    gen_cases = [(inp, out) for inp, out in gen_cases if inp is not None and out is not None]

    # ── test_bool_table: K_CODE × n_gt ───────────────────────────────
    n_gt = min(len(test_inputs), MAX_TEST)
    test_bool_table = []
    for code in codes:
        if code is None:
            test_bool_table.append([False] * n_gt)
            continue
        row = [
            outputs_match(run_code(code, inp, timeout=time_limit), exp)
            for inp, exp in zip(test_inputs[:n_gt], test_outputs[:n_gt])
        ]
        test_bool_table.append(row)

    # ── case_bool_table: K_CODE × K_CASE_valid ───────────────────────
    case_bool_table = []
    for code in codes:
        if code is None or not gen_cases:
            case_bool_table.append([False] * len(gen_cases))
            continue
        row = [
            outputs_match(run_code(code, c_inp, timeout=time_limit), c_exp)
            for c_inp, c_exp in gen_cases
        ]
        case_bool_table.append(row)

    result = {
        "codes":           codes,
        "gen_cases":       gen_cases,
        "test_bool_table": test_bool_table,
        "case_bool_table": case_bool_table,
    }
    ut_results[prob_idx] = result

    # ── save checkpoint ───────────────────────────────────────────────
    checkpoint[prob_idx] = {
        "codes":           codes,
        "gen_cases":       [list(x) for x in gen_cases],
        "test_bool_table": test_bool_table,
        "case_bool_table": case_bool_table,
    }
    _save_checkpoint(CHECKPOINT_PATH, checkpoint)

    # ── per-problem summary ───────────────────────────────────────────
    n_correct_code = sum(1 for row in test_bool_table if row and all(row))
    n_valid_cases  = len(gen_cases)
    print(
        f"[{prob_idx+1:3d}/{len(UT_PROBLEMS)}]  "
        f"correct code {n_correct_code}/{K_CODE}  "
        f"valid cases {n_valid_cases}/{K_CASE}  "
        f"(code prompt {code_prompt_len} tok)"
    )

In [ ]:
from math import comb
import statistics

valid_results = [r for r in ut_results if r is not None]
valid_probs   = [p for p, r in zip(UT_PROBLEMS, ut_results) if r is not None]
n = len(valid_results)

# ── pass@k (unbiased estimator, Chen et al. 2021) ────────────────────────
# pass@k = E_problems[1 - C(n-c, k) / C(n, k)]  where n=K_CODE, c=correct
# Error bars: 95% CI via CLT — each problem contributes an independent x_i,
# SE = std(x_i) / sqrt(n),  CI = ±1.96 * SE
def _pass_at_k(n_samples, n_correct, k):
    if n_samples - n_correct < k:
        return 1.0
    return 1.0 - comb(n_samples - n_correct, k) / comb(n_samples, k)

_ks = sorted({1} | {2**i for i in range(1, 32) if 2**i <= K_CODE} | {K_CODE})

_correct_counts = [
    sum(1 for row in r["test_bool_table"] if row and all(row))
    for r in valid_results
]

pass_at_k = {}
pass_at_k_ci = {}
for k in _ks:
    per_problem = [_pass_at_k(K_CODE, c, k) for c in _correct_counts]
    mean = sum(per_problem) / n if n else 0.0
    se   = (statistics.stdev(per_problem) / n**0.5) if n > 1 else 0.0
    pass_at_k[k]    = mean
    pass_at_k_ci[k] = 1.96 * se

# ── BoN_acc: case_bool_table selects best code → check on gt tests ────────
bon_correct = 0
for r in valid_results:
    if not r["gen_cases"] or not any(r["case_bool_table"]):
        best_idx = 0
    else:
        case_scores = [sum(row) for row in r["case_bool_table"]]
        best_idx    = case_scores.index(max(case_scores))
    test_row = r["test_bool_table"][best_idx]
    if test_row and all(test_row):
        bon_correct += 1

# ── case_quality via correct solver code ─────────────────────────────────
cq_pass = cq_total = 0
for prob, r in zip(valid_probs, valid_results):
    time_limit = float(prob.get("test_time_limit", 2.0))
    oracle_code = next(
        (code for code, row in zip(r["codes"], r["test_bool_table"])
         if code is not None and row and all(row)),
        None,
    )
    if oracle_code is None:
        continue
    for inp, tester_exp in r["gen_cases"]:
        oracle_out = run_code(oracle_code, inp, timeout=time_limit)
        if oracle_out == "timeout" or oracle_out.startswith("error:"):
            continue
        cq_pass  += int(outputs_match(oracle_out, tester_exp))
        cq_total += 1

# ── p_01, p_00 ────────────────────────────────────────────────────────────
p01_score = p01_num = 0
p00_score = p00_num = 0
for r in valid_results:
    n_codes = len(r["test_bool_table"])
    n_cases = len(r["gen_cases"])
    correct_code_idx = [i for i, row in enumerate(r["test_bool_table"]) if row and all(row)]
    wrong_code_idx   = [i for i in range(n_codes) if i not in correct_code_idx]
    if not correct_code_idx or n_cases == 0:
        continue
    correct_case_idx = [
        j for j in range(n_cases)
        if all(r["case_bool_table"][i][j] for i in correct_code_idx)
    ]
    wrong_case_idx = [j for j in range(n_cases) if j not in correct_case_idx]
    for wi in wrong_code_idx:
        for cj in correct_case_idx:
            p01_score += 1 - int(r["case_bool_table"][wi][cj])
            p01_num   += 1
    for wi in wrong_code_idx:
        for wj in wrong_case_idx:
            p00_score += int(r["case_bool_table"][wi][wj])
            p00_num   += 1

def _pct(a, b): return f"{a/b:.1%}" if b else "N/A"

print(f"Problems evaluated : {n}  ({ut_skipped} skipped — prompt too long)")
for k in _ks:
    print(f"pass@{k:<3}            : {pass_at_k[k]:.1%} ± {pass_at_k_ci[k]:.1%}  (95% CI)")
print(f"BoN_acc            : {bon_correct}/{n}  ({_pct(bon_correct, n)})  unit-test-selected code passes all gt tests")
print(f"case_quality       : {cq_pass}/{cq_total}  ({_pct(cq_pass, cq_total)})  tester output vs correct solver code  (problems w/o correct code skipped)")
print(f"p_01 (discrimin.)  : {_pct(p01_score, p01_num)}  correct cases that reject wrong code  (higher = better)")
print(f"p_00 (leakage)     : {_pct(p00_score, p00_num)}  wrong cases that wrong code still passes  (lower = better)")

In [ ]:
# ── Extract verified solver codes from unit-test eval ─────────────────────
# Use the first code per problem that passed all GT tests as the oracle for
# tester accuracy in the iterative eval.  Avoids running human-written GT
# solutions through run_code (which can crash the kernel via bad imports or
# exec side-effects).
#
# NOTE: UT_PROBLEMS should cover at least the same slice as ITER_PROBLEMS.
# Problems in ITER_PROBLEMS that are not in UT_PROBLEMS will have no verified
# solution; tester_vs_gt_per_turn will be empty ([]) for those problems.
verified_solutions = {}
for prob, result in zip(UT_PROBLEMS, ut_results):
    if result is None:
        continue
    task_id = prob.get("task_id")
    if task_id is None:
        continue
    correct_code = next(
        (code for code, row in zip(result["codes"], result["test_bool_table"])
         if code is not None and row and all(row)),
        None,
    )
    if correct_code is not None:
        verified_solutions[str(task_id)] = correct_code

print(f"Verified solutions: {len(verified_solutions)}/{len(UT_PROBLEMS)} problems have ≥1 correct code")

## Iterative refinement eval

Multi-turn diagnostic: solver generates code, tester generates targeted test cases against that code, solver refines based on feedback. Repeats for `N_TURNS` turns.

Tracks per-turn GT accuracy to measure whether the feedback loop helps or hurts.

In [ ]:
import importlib
importlib.reload(iterative_eval)   # iterative_eval imported in cell 71780324; reload picks up edits
# After reloading, re-run cell 71780324 to refresh the `from iterative_eval import ...` bindings.

# ── config ────────────────────────────────────────────────────────────────────
N_TURNS       = 2    # refinement turns (not counting turn 0)
K_CASE_ITER   = 3    # test cases tester generates per turn (single LLM call)
HISTORY_TURNS = 1    # previous solver turns kept in context (1 = latest only)
ITER_PROBLEMS = data[:40]
VERBOSE       = False  # True: print every prompt + response for manual inspection

# Set to a file path to enable save/resume; None to disable.
ITER_CHECKPOINT_PATH = "/tmp/iter_eval_checkpoint.json"

In [ ]:
_skip = {i for i in range(len(ITER_PROBLEMS)) if i in FORCE_SKIP}
_to_run = [(i, p) for i, p in enumerate(ITER_PROBLEMS) if i not in _skip]

if _skip:
    print(f"Force-skipping {len(_skip)} problem(s): {sorted(_skip)}")

_raw_results = iterative_eval.run_iterative_eval(
    sampler         = sampler,
    tokenizer       = tokenizer,
    problems        = [p for _, p in _to_run],
    n_turns         = N_TURNS,
    k_case          = K_CASE_ITER,
    max_new_tokens  = MAX_NEW_TOKENS,
    max_prompt_len  = MAX_PROMPT_LEN,
    temperature     = TEMPERATURE,
    top_p           = TOP_P,
    exec_timeout    = EXEC_TIMEOUT,
    max_gt_test     = MAX_TEST,
    history_turns   = HISTORY_TURNS,
    gt_solutions    = verified_solutions,   # verified solver codes from unit-test eval
    use_gt_oracle   = True,
    checkpoint_path = ITER_CHECKPOINT_PATH,
    verbose         = VERBOSE,
)

# Reconstruct full list aligned with ITER_PROBLEMS (None for skipped slots).
_raw_iter = iter(_raw_results)
iter_results = [
    None if i in _skip else next(_raw_iter)
    for i in range(len(ITER_PROBLEMS))
]

In [ ]:
valid = [r for r in iter_results if r is not None]
n     = len(valid)
T     = N_TURNS

def _pct(a, b): return f"{a/b:.1%}" if b else "N/A"

# ── pass@turn_t trajectory ────────────────────────────────────────────────────
print(f"{'Turn':>5}  {'pass':>6}  {'pct':>7}")
print("-" * 24)
for t in range(T + 1):
    n_pass = sum(1 for r in valid if r["gt_pass_per_turn"][t])
    print(f"  t={t}  {n_pass:>3}/{n:<3}  {_pct(n_pass, n):>7}")

# ── improvement / regression ──────────────────────────────────────────────────
failed_0  = [r for r in valid if not r["gt_pass_per_turn"][0]]
passed_0  = [r for r in valid if     r["gt_pass_per_turn"][0]]
improved  = sum(1 for r in failed_0 if r["gt_pass_per_turn"][T])
regressed = sum(1 for r in passed_0 if not r["gt_pass_per_turn"][T])

print(f"\nimprovement_rate  {improved}/{len(failed_0)}  {_pct(improved, len(failed_0))}  "
      f"(failed t=0, pass t={T})")
print(f"regression_rate   {regressed}/{len(passed_0)}  {_pct(regressed, len(passed_0))}  "
      f"(passed t=0, fail t={T})  ← tester-misled-solver diagnostic")

# ── case_quality via GT solutions ─────────────────────────────────────────────
# tester_vs_gt_per_turn[t][i] is True if the tester's predicted output for
# generated test i at turn t matches the GT solution's output.
# Only populated for problems whose task_id appears in gt_solutions.
cq_pass = cq_total = 0
for r in valid:
    for turn_matches in r["tester_vs_gt_per_turn"]:
        cq_pass  += sum(turn_matches)
        cq_total += len(turn_matches)

print(f"\ncase_quality  {cq_pass}/{cq_total}  {_pct(cq_pass, cq_total)}  "
      f"tester predicted output vs GT solution  (low → tester expected outputs are wrong)")

## Oracle tester iterative refinement eval

Same multi-turn solver loop as above, but the **tester is replaced by ground-truth test cases** — so the expected outputs shown to the solver are guaranteed correct.

Pipeline per turn:
1. Run the solver's current code against all GT test cases (up to `MAX_TEST`).
2. If **all pass** → tell the solver its code is correct and stop early.
3. If **some fail** → randomly sample up to `MAX_FAILURES_SHOWN` of the failures, show the solver each failure's input, its actual output, and the guaranteed-correct expected output.
4. Solver refines its code based on that feedback.

The solver prompt omits the "the tester may be wrong" caveat used in the LLM-tester variant.

In [ ]:
import importlib
importlib.reload(iterative_eval)
from iterative_eval import run_oracle_iterative_eval

# ── config ───────────────────────────────────────────────────────────────────
ORACLE_N_TURNS          = 2    # refinement turns (not counting turn 0); compare against pass@(N_TURNS+1)
ORACLE_MAX_FAILURES     = 3    # max failed cases shown per turn (randomly sampled)
ORACLE_HISTORY_TURNS    = ORACLE_N_TURNS  # keep full history for cleanest pass@K comparison
ORACLE_PROBLEMS         = data[:40]
ORACLE_VERBOSE          = False

ORACLE_CHECKPOINT_PATH  = "/tmp/oracle_iter_eval_checkpoint.json"

In [ ]:
_oracle_skip   = {i for i in range(len(ORACLE_PROBLEMS)) if i in FORCE_SKIP}
_oracle_to_run = [(i, p) for i, p in enumerate(ORACLE_PROBLEMS) if i not in _oracle_skip]

if _oracle_skip:
    print(f"Force-skipping {len(_oracle_skip)} problem(s): {sorted(_oracle_skip)}")

_oracle_raw = run_oracle_iterative_eval(
    sampler              = sampler,
    tokenizer            = tokenizer,
    problems             = [p for _, p in _oracle_to_run],
    n_turns              = ORACLE_N_TURNS,
    max_failures_shown   = ORACLE_MAX_FAILURES,
    max_new_tokens       = MAX_NEW_TOKENS,
    max_prompt_len       = MAX_PROMPT_LEN,
    temperature          = TEMPERATURE,
    top_p                = TOP_P,
    exec_timeout         = EXEC_TIMEOUT,
    max_gt_test          = MAX_TEST,
    history_turns        = ORACLE_HISTORY_TURNS,
    checkpoint_path      = ORACLE_CHECKPOINT_PATH,
    verbose              = ORACLE_VERBOSE,
)

# Reconstruct full list aligned with ORACLE_PROBLEMS (None for force-skipped slots).
_oracle_iter = iter(_oracle_raw)
oracle_results = [
    None if i in _oracle_skip else next(_oracle_iter)
    for i in range(len(ORACLE_PROBLEMS))
]

In [ ]:
valid  = [r for r in oracle_results if r is not None]
n      = len(valid)
T      = ORACLE_N_TURNS
skipped = sum(1 for r in oracle_results if r is None)

def _pct(a, b): return f"{a/b:.1%}" if b else "N/A"

def _turn_pass(r, t):
    hist = r["gt_pass_per_turn"]
    if t < len(hist):
        return hist[t]
    return hist[-1]  # early-stopped while passing

def _ever_trimmed(r):
    return any(r.get("trimmed_per_turn", []))

# ── trimming diagnostic ───────────────────────────────────────────────────────
n_trimmed = sum(1 for r in valid if _ever_trimmed(r))
if n_trimmed:
    print(f"WARNING: {n_trimmed}/{n} problems had history trimmed to fit context.")
    print(f"  These problems saw less history than pass@K baselines, making the")
    print(f"  comparison unfair from that turn onwards. Consider reducing N_TURNS")
    print(f"  or MAX_PROMPT_LEN to avoid trimming.\n")
else:
    print(f"No trimming occurred — full history was preserved for all problems.\n")

# ── pass@turn trajectory (trimmed problems flagged) ───────────────────────────
print(f"{'Turn':>5}  {'pass':>6}  {'pct':>7}  {'trimmed':>8}")
print("-" * 36)
for t in range(1, T + 1):
    n_pass    = sum(1 for r in valid if _turn_pass(r, t))
    n_trim_t  = sum(
        1 for r in valid
        if t - 1 < len(r.get("trimmed_per_turn", [])) and r["trimmed_per_turn"][t - 1]
    )
    trim_flag = f"  ← {n_trim_t} trimmed" if n_trim_t else ""
    print(f"  t={t}  {n_pass:>3}/{n:<3}  {_pct(n_pass, n):>7}{trim_flag}")

# also print t=0 baseline (no feedback yet, no trimming possible)
n_pass_0 = sum(1 for r in valid if _turn_pass(r, 0))
print(f"\n  t=0  {n_pass_0:>3}/{n:<3}  {_pct(n_pass_0, n):>7}  (baseline, no feedback)")

# ── improvement / regression ──────────────────────────────────────────────────
failed_0  = [r for r in valid if not _turn_pass(r, 0)]
passed_0  = [r for r in valid if     _turn_pass(r, 0)]
improved  = sum(1 for r in failed_0 if _turn_pass(r, T))
regressed = sum(1 for r in passed_0 if not _turn_pass(r, T))

print(f"\nimprovement_rate  {improved}/{len(failed_0)}  {_pct(improved, len(failed_0))}"
      f"  (failed t=0, pass t={T})")
print(f"regression_rate   {regressed}/{len(passed_0)}  {_pct(regressed, len(passed_0))}"
      f"  (passed t=0, fail t={T})")
print(f"\n({skipped} problem(s) skipped — prompt too long)")